In [2]:
import rasterio
import numpy as np

# Reference grid
reference = r"D:\landslide\final_data\distance_roads_tehri.tif"

# List ALL aligned rasters (10m versions only)
rasters = [
    r"D:\landslide\final_data\Elevation_10m.tif",
    r"D:\landslide\final_data\Slope_deg_10m.tif",
    r"D:\landslide\final_data\Aspect_deg_10m.tif",
    r"D:\landslide\final_data\Curvature_10m.tif",
    r"D:\landslide\final_data\TRI_10m.tif",
    r"D:\landslide\final_data\TWI_clean_10m.tif",
    r"D:\landslide\final_data\Rainfall_Tehri_clean_10m.tif",
    r"D:\landslide\final_data\distance_streams_tehri.tif",
    r"D:\landslide\final_data\distance_roads_tehri.tif",
    r"D:\landslide\final_data\distance_faults_tehri.tif",
    r"D:\landslide\final_data\NDVI_Tehri_10m.tif",
    r"D:\landslide\final_data\BSI_lite_Tehri_10m.tif",
    r"D:\landslide\final_data\Soil_Tehri_10m.tif",
    r"D:\landslide\final_data\lithology_tehri_10m.tif",
    r"D:\landslide\final_data\lulc_tehri_10m.tif",
    r"D:\landslide\final_data\Geomorphon_clean_10m.tif"
]

with rasterio.open(reference) as ref:
    meta = ref.meta.copy()
    height, width = ref.height, ref.width

meta.update(count=len(rasters), dtype='float32')

stack = np.zeros((len(rasters), height, width), dtype=np.float32)

for i, path in enumerate(rasters):
    with rasterio.open(path) as src:
        stack[i, :, :] = src.read(1)

output_stack = r"D:\landslide\landslide_raster\LSM_stack_10m.tif"

with rasterio.open(output_stack, "w", **meta) as dst:
    dst.write(stack)

print("Stack created successfully.")

Stack created successfully.


In [1]:
import geopandas as gpd

# Paths
landslide_path = r"D:\landslide\inventory\Landslide Polygon.shp"
tehri_boundary_path = r"F:\New folder (2)\tehri_new\tehri.shp"
output_path = r"D:\landslide\inventory\landslide_polygons_tehri.shp"

# Load data
landslides = gpd.read_file(landslide_path)
tehri = gpd.read_file(tehri_boundary_path)

# Ensure same CRS
if landslides.crs != tehri.crs:
    landslides = landslides.to_crs(tehri.crs)

# Clip
clipped = gpd.clip(landslides, tehri)

# Save
clipped.to_file(output_path)

print("Polygons clipped successfully.")
print("Original count:", len(landslides))
print("Clipped count:", len(clipped))

Polygons clipped successfully.
Original count: 7256
Clipped count: 2613


In [2]:
points_path = r"D:\landslide\inventory\Landslide Point.shp"
output_points = r"D:\landslide\land_inv\landslide_points_tehri.shp"

points = gpd.read_file(points_path)

if points.crs != tehri.crs:
    points = points.to_crs(tehri.crs)

clipped_points = gpd.clip(points, tehri)
clipped_points.to_file(output_points)

print("Points clipped successfully.")
print("Original:", len(points))
print("Clipped:", len(clipped_points))

Points clipped successfully.
Original: 2536
Clipped: 710


In [3]:
import geopandas as gpd

# Paths
polygon_path = r"D:\landslide\land_inv\landslide_polygons_tehri.shp"
points_path = r"D:\landslide\land_inv\landslide_points_tehri.shp"
boundary_path = r"F:\New folder (2)\tehri_new\tehri.shp"

# Load
polygons = gpd.read_file(polygon_path)
points = gpd.read_file(points_path)
boundary = gpd.read_file(boundary_path)

# Print CRS
print("Polygons CRS:", polygons.crs)
print("Points CRS:", points.crs)
print("Boundary CRS:", boundary.crs)

Polygons CRS: EPSG:32644
Points CRS: EPSG:32644
Boundary CRS: EPSG:32644


In [4]:
print("Polygons after clipping:", len(clipped))
print("Points after clipping:", len(clipped_points))

Polygons after clipping: 2613
Points after clipping: 710


In [5]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import numpy as np

# Paths
polygon_path = r"D:\landslide\land_inv\landslide_polygons_tehri.shp"
reference_raster = r"D:\landslide\final_data\distance_roads_tehri.tif"
output_mask = r"D:\landslide\final_data\landslide_mask_10m.tif"

# Load polygons
gdf = gpd.read_file(polygon_path)

# Open reference raster
with rasterio.open(reference_raster) as ref:
    meta = ref.meta.copy()
    transform = ref.transform
    out_shape = (ref.height, ref.width)

# Rasterize
mask = rasterize(
    [(geom, 1) for geom in gdf.geometry],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype='uint8'
)

# Save mask
meta.update(count=1, dtype='uint8')

with rasterio.open(output_mask, 'w', **meta) as dst:
    dst.write(mask, 1)

print("Landslide mask created successfully.")

Landslide mask created successfully.


In [9]:
path = r"D:\landslide\landslide_raster\landslide_mask_10m.tif"
with rasterio.open(path) as src:
    data = src.read(1)
    print("Unique values:", np.unique(data))

Unique values: [0 1]


In [10]:
import rasterio
import numpy as np

mask_path = r"D:\landslide\landslide_raster\landslide_mask_10m.tif"

with rasterio.open(mask_path) as src:
    mask = src.read(1)

landslide_pixels = np.sum(mask == 1)
non_landslide_pixels = np.sum(mask == 0)

print("Landslide pixels:", landslide_pixels)
print("Non-landslide pixels:", non_landslide_pixels)

Landslide pixels: 35923
Non-landslide pixels: 97535321


In [3]:
stack_path = r"D:\landslide\landslide_raster\LSM_stack_10m.tif"
mask_path = r"D:\landslide\landslide_raster\landslide_mask_10m.tif"

# Read data
with rasterio.open(stack_path) as src:
    stack = src.read()  # shape: (bands, rows, cols)

with rasterio.open(mask_path) as src:
    mask = src.read(1)

bands, rows, cols = stack.shape

# Reshape stack
stack_reshaped = stack.reshape(bands, -1).T   # shape: (pixels, bands)
mask_flat = mask.flatten()

# Landslide pixels
ls_idx = np.where(mask_flat == 1)[0]

# Non-landslide pixels
nls_idx = np.where(mask_flat == 0)[0]

# Randomly sample non-landslide equal to landslide
np.random.seed(42)
nls_sample = np.random.choice(nls_idx, size=len(ls_idx), replace=False)

# Combine indices
final_idx = np.concatenate([ls_idx, nls_sample])

# Extract features
X = stack_reshaped[final_idx]
y = mask_flat[final_idx]

In [4]:
import pandas as pd
# Create dataframe
df = pd.DataFrame(X)
df["label"] = y

print("Dataset shape:", df.shape)
print(df["label"].value_counts())

Dataset shape: (71846, 17)
label
1    35923
0    35923
Name: count, dtype: int64


In [7]:
stack_path = r"D:\landslide\landslide_raster\LSM_stack_10m.tif"

with rasterio.open(stack_path) as src:
    print("Number of bands:", src.count)
    
    for i in range(1, src.count + 1):
        print(f"Band {i} description:", src.descriptions[i-1])

Number of bands: 16
Band 1 description: None
Band 2 description: None
Band 3 description: None
Band 4 description: None
Band 5 description: None
Band 6 description: None
Band 7 description: None
Band 8 description: None
Band 9 description: None
Band 10 description: None
Band 11 description: None
Band 12 description: None
Band 13 description: None
Band 14 description: None
Band 15 description: None
Band 16 description: None


In [12]:
feature_names = [
    "elevation",
    "slope",
    "aspect",
    "curvature",
    "TRI",
    "TWI",
    "rainfall",
    "dist_stream",
    "dist_road",
    "dist_fault",
    "NDVI",
    "BSI",
    "soil",
    "lithology",
    "LULC",
    "geomorphon"
]

df.columns = feature_names + ["label"]

print(df.head())

     elevation      slope      aspect  curvature        TRI        TWI  \
0  3964.060547  53.417248  308.530182  -0.000697  23.812599  10.389105   
1  3955.881348  53.340611  308.530182  -0.001078  23.698778  10.507830   
2  3947.371826  52.557346  311.633545  -0.001786  23.103390  10.388711   
3  3960.437500  52.179382  306.869904  -0.001116  22.586826   9.671568   
4  3952.341553  52.095924  308.530182  -0.001254  22.354893  10.597772   

   rainfall   dist_stream     dist_road    dist_fault      NDVI       BSI  \
0   -9999.0  27237.910156  29796.917969  89959.109375  0.100897  0.214427   
1   -9999.0  27247.183594  29806.853516  89969.101562  0.129019  0.175965   
2   -9999.0  27256.457031  29816.787109  89979.101562  0.132021  0.170064   
3   -9999.0  27232.380859  29788.134766  89948.968750  0.046843  0.273991   
4   -9999.0  27241.652344  29798.068359  89958.968750  0.082975  0.236869   

     soil  lithology  LULC  geomorphon  label  
0  3717.0        1.0  11.0         7.0      

In [13]:
df.to_csv("LSM_Data.csv", index=False)

In [14]:
import pandas as pd
df = pd.read_csv("LSM_Data.csv")
print(df.head())

   elevation      slope     aspect  curvature        TRI        TWI  rainfall  \
0  3964.0605  53.417248  308.53018  -0.000697  23.812600  10.389105   -9999.0   
1  3955.8813  53.340610  308.53018  -0.001078  23.698778  10.507830   -9999.0   
2  3947.3718  52.557346  311.63354  -0.001786  23.103390  10.388711   -9999.0   
3  3960.4375  52.179382  306.86990  -0.001116  22.586826   9.671568   -9999.0   
4  3952.3416  52.095924  308.53018  -0.001254  22.354893  10.597772   -9999.0   

   dist_stream  dist_road  dist_fault      NDVI       BSI    soil  lithology  \
0    27237.910  29796.918    89959.11  0.100897  0.214427  3717.0        1.0   
1    27247.184  29806.854    89969.10  0.129019  0.175965  3717.0        1.0   
2    27256.457  29816.787    89979.10  0.132021  0.170064  3717.0        1.0   
3    27232.380  29788.135    89948.97  0.046843  0.273991  3717.0        1.0   
4    27241.652  29798.068    89958.97  0.082975  0.236869  3717.0        1.0   

   LULC  geomorphon  label  
0  

In [15]:
df.value_counts("geomorphon")

geomorphon
 6.0       22109
-9999.0    19655
 7.0       11997
 5.0        8552
 9.0        5647
 3.0        2374
 10.0       1138
 2.0         246
 8.0          65
 4.0          62
 1.0           1
Name: count, dtype: int64

In [16]:
df.value_counts("rainfall")

rainfall
-9999.0000    55857
 1104.6549        2
 1160.2750        2
 880.9849         2
 1183.1906        2
              ...  
 1413.2731        1
 1415.7632        1
 1415.9421        1
 1416.0520        1
-9873.8110        1
Name: count, Length: 15961, dtype: int64

In [17]:
import numpy as np
df = df.replace(-9999, np.nan)

In [18]:
print(df.isna().sum())

elevation      19655
slope          19695
aspect         19851
curvature      19655
TRI            19599
TWI            24269
rainfall       55857
dist_stream        0
dist_road          0
dist_fault         0
NDVI           19607
BSI            19607
soil               0
lithology          0
LULC               0
geomorphon     19655
label              0
dtype: int64


In [19]:
df = df.dropna()

In [20]:
print(df["label"].value_counts())

label
1    9358
0    5405
Name: count, dtype: int64


In [21]:
df

,elevation,slope,aspect,curvature,TRI,TWI,rainfall,dist_stream,dist_road,dist_fault,NDVI,BSI,soil,lithology,LULC,geomorphon,label
1355,2682.3113,36.920950,322.59464,0.000488,13.474232,9.929188,248.097050,25502.7070,25473.352,82576.06,0.401523,-0.079778,3717.0,1.0,2.0,6.0,1
1356,2676.3433,37.650290,322.59464,0.000386,13.635516,9.833165,243.829280,25511.8050,25482.307,82586.03,0.398395,-0.075737,3717.0,1.0,2.0,6.0,1
1357,2671.1135,37.497524,311.05480,0.000376,13.393378,9.577889,239.561540,25520.9020,25491.262,82596.01,0.410995,-0.086975,3717.0,1.0,2.0,6.0,1
1358,2697.0974,33.080550,327.99463,0.000915,12.088349,9.970140,260.922850,25479.5780,25450.950,82545.38,0.309149,0.023426,3717.0,1.0,2.0,6.0,1
1359,2691.9130,35.451572,327.99463,0.000811,12.990210,9.728001,256.655100,25488.6720,25459.902,82555.35,0.310834,0.018931,3717.0,1.0,2.0,6.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71825,1258.3613,29.291826,88.31531,-0.001024,10.559682,9.367873,562.867000,6680.0073,28189.750,49534.75,0.433115,-0.091276,3661.0,8.0,2.0,7.0,0
71830,2343.9314,32.783615,343.61047,0.001820,11.963850,9.941607,418.881100,29136.1230,29302.627,81977.80,0.473003,-0.147851,3717.0,2.0,2.0,7.0,0
71836,1283.1699,33.411230,296.56506,-0.004185,11.806582,7.940690,145.517170,7770.0000,60202.000,56442.83,0.443750,-0.115644,3661.0,3.0,11.0,6.0,0
71843,1743.3475,27.919088,285.94540,0.016986,10.429208,11.511685,1337.446200,13177.4960,55571.168,50884.15,0.107248,0.240932,3661.0,4.0,2.0,5.0,0


In [22]:
df.to_csv("LSM_Data_clean.csv", index=False)